In [20]:
import json

objects = json.load(open("../objects/objects.json", "r"))


def makedsodict(thiscatalog, ifirst, ilast):
    dsodict = {}
    for i in range(ifirst, ilast + 1):
        for id in objects:
            object = objects[id]
            if "Catalog %s" % thiscatalog not in object:
                continue
            if object["Catalog %s" % thiscatalog] != i:
                continue

            if thiscatalog in ["C", "M", "U", "BSC"]:
                thisname = "%d" % object["Catalog %s" % thiscatalog]
            else:
                thisname = "%s %d" % (thiscatalog, object["Catalog %s" % thiscatalog])

            if "Catalog NGC" in object and isinstance(object["Catalog NGC"], int):
                starplotname = "NGC%04d" % object["Catalog NGC"]
            elif "Catalog IC" in object and isinstance(object["Catalog IC"], int):
                starplotname = "IC%04d" % object["Catalog IC"]
            elif "Catalog M" in object:
                starplotname = "M%03d" % object["Catalog M"]
            else:
                print("cannot find dso for %s %d (%s)" % (thiscatalog, i, id))
                starplotname = None
            if starplotname != None:
                dsodict[starplotname] = thisname
    return dsodict

In [21]:
import starplot
from starplot.styles import PlotStyle, extensions

style = PlotStyle().extend(
    #extensions.BLUE_LIGHT,
    extensions.MAP,
)


for objectstyle in [
    style.dso_association_stars,
    style.dso_dark_nebula,
    style.dso_double_star,
    style.dso_duplicate,
    style.dso_galaxy,
    style.dso_globular_cluster,
    style.dso_nebula,
    style.dso_nonexistant,
    style.dso_nova_star,
    style.dso_open_cluster,
    style.dso_planetary_nebula,
    style.dso_supernova_remnant,
]:
    objectstyle.marker.fill = "rgba(0, 0, 0, 0.07)"
    objectstyle.marker.stroke_width = 2
    objectstyle.label.font_size = 36

def makeplot(prefix, thiscatalog, ifirst, ilast):

    dsodict = makedsodict(thiscatalog, ifirst, ilast)
    dsonames = dsodict.keys()

    p = starplot.MapPlot(
        projection=starplot.Robinson(),
        ra_min=0,
        ra_max=360,
        dec_min=-90,
        dec_max=90,
        style=style,
        resolution=6000,
        # since this map has a very large extent, let's scale everything down
        autoscale=True,
    )
    p.gridlines(labels=False)
    p.constellations()
    #p.constellation_labels(
    #    style__font_size=28,
    #    style__opacity=0.5,
    #)
    # p.constellation_borders()
    p.milky_way()
    # p.celestial_equator(label="")
    p.stars(
        where=[starplot._.magnitude < 5],
        where_labels=False,
        opacity_fn=lambda star: 0.3,
        size_fn=lambda star: (6 - star.magnitude) * 3,
    )

    p.dsos(
        where=[
            starplot._.name.isin(dsonames),
        ],
        where_labels=[True],
        where_true_size=[False],
        collision_handler=starplot.CollisionHandler(
            plot_on_fail=True,
            attempts=100,
            allow_constellation_line_collisions=True,
            allow_marker_collisions=True,
            anchor_fallbacks=[
                starplot.AnchorPoint.BOTTOM_RIGHT,
                starplot.AnchorPoint.TOP_LEFT,
                starplot.AnchorPoint.TOP_RIGHT,
                starplot.AnchorPoint.BOTTOM_LEFT,
                starplot.AnchorPoint.BOTTOM_CENTER,
                starplot.AnchorPoint.TOP_CENTER,
                starplot.AnchorPoint.RIGHT_CENTER,
                starplot.AnchorPoint.LEFT_CENTER,
            ],
        ),
        label_fn=lambda dso: dsodict[dso.name],
    )

    p.export("%s/%s-allsky.svg" % (prefix, thiscatalog))


makeplot("../finder/figures", "M", 1, 110)
makeplot("../finder/figures", "C", 1, 110)
makeplot("../finder/figures", "U", 1, 87)
makeplot("../binocular/figures", "BSC", 1, 150)


cannot find dso for C 9 (C9)
cannot find dso for C 14 (C14)
cannot find dso for C 41 (C41)
cannot find dso for C 49 (C49)
cannot find dso for C 99 (C99)
cannot find dso for C 100 (C100)
cannot find dso for U 6 (U6)
cannot find dso for U 8 (U8)
cannot find dso for U 11 (U11)
cannot find dso for U 13 (U13)
cannot find dso for U 14 (U14)
cannot find dso for U 15 (U15)
cannot find dso for U 18 (C41)
cannot find dso for U 44 (U44)
cannot find dso for U 72 (U72)
cannot find dso for BSC 1 (C41)
cannot find dso for BSC 22 (C100)
cannot find dso for BSC 96 (C9)
cannot find dso for BSC 135 (C49)


In [22]:
!sh svgtopdf.sh ../finder/figures/*-allsky.svg
!sh svgtopdf.sh ../binocular/figures/*-allsky.svg

C-allsky.pdf
M-allsky.pdf
U-allsky.pdf
BSC-allsky.pdf
